# Target 2 — Load/Demand (item_count_sum) Forecast — 7-Day
**AstroLogix Logistics Forecasting System**

## What are we doing?
We forecast the **item_count** column sum per region × day from `orders.parquet` as a load/demand proxy.

## Why this approach?
Per the project lead's note: *"Target 2 is basically order count × average weight"* — so we solve it with the same pipeline, just a different target.

## Pipeline Summary
```
orders.parquet (timestamp → hour/day/month/year extraction)
    ↓
Hourly pattern → daily aggregate features
    ↓
Holidays + Weather data
    ↓
Full Feature Engineering (leak-free, shift(1))
    ↓
Time-based train/test split (NO shuffle!)
    ↓
6 ML models → comparison → best selected
    ↓
Retrain on full history → recursive 7-day forecast
```

In [1]:
import pandas as pd
import numpy as np
import warnings
import random
import joblib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    KFold,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor
)

from sklearn.svm import SVR

from sklearn.neighbors import KNeighborsRegressor

from sklearn.tree import DecisionTreeRegressor
import bisect
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")

print("✅ All libraries loaded successfully.")

✅ All libraries loaded successfully.


## 1. Data Loading — Load Parquet Files

In [2]:
orders   = pd.read_parquet('orders.parquet')
holidays = pd.read_parquet('holidays.parquet')
weather  = pd.read_parquet('weather.parquet')

# created_at → datetime (strip timezone offset)
orders['created_at'] = pd.to_datetime(
    orders['created_at'].astype(str).str.replace(r'\+00:00$', '', regex=True)
)

print(f"Orders   : {orders.shape}  |  columns: {list(orders.columns)}")
print(f"Holidays : {holidays.shape}")
print(f"Weather  : {weather.shape}")
print(f"Date range: {orders['created_at'].min()} -> {orders['created_at'].max()}")
print(f"Regions  : {sorted(orders['region'].unique())}")


Orders   : (50000, 6)  |  columns: ['order_id', 'region', 'item_count', 'created_at', 'delivery_type', 'shipment_id']
Holidays : (135, 3)
Weather  : (23570, 5)
Date range: 2020-01-01 02:39:46 -> 2026-06-12 22:13:48
Regions  : ['Absheron', 'Ganja', 'Kalbajar', 'Khachmaz', 'Lankaran', 'Nakhchivan', 'Qazakh', 'Sheki', 'Yevlakh']


## 2. Timestamp Decomposition — Hour, Day, Month, Year Features

The `created_at` column contains both date and time information.  
We extract **hour**-based behaviour features from each order, then aggregate them as daily columns.

**Why hour features?**  
- Orders arriving at fixed hours (e.g. lunch, evening) reveal a region's logistics profile  
- The share of night orders signals express delivery demand  
- "Order hour standard deviation" shows how spread out demand is throughout the day  

⚠️ These features will be converted to **daily sums**, then lagged with **shift(1)** — no leakage.

In [3]:
# ── Extract time components from timestamp ────────────────────────────────────
orders['date']   = orders['created_at'].dt.normalize()
orders['hour']   = orders['created_at'].dt.hour
orders['minute'] = orders['created_at'].dt.minute

# Intra-day time-of-day buckets
orders['is_early_morning']  = ((orders['hour'] >= 5)  & (orders['hour'] < 9)).astype(int)
orders['is_morning']        = ((orders['hour'] >= 9)  & (orders['hour'] < 12)).astype(int)
orders['is_afternoon']      = ((orders['hour'] >= 12) & (orders['hour'] < 17)).astype(int)
orders['is_evening']        = ((orders['hour'] >= 17) & (orders['hour'] < 21)).astype(int)
orders['is_night']          = ((orders['hour'] >= 21) | (orders['hour'] < 5)).astype(int)
orders['is_business_hours'] = ((orders['hour'] >= 9)  & (orders['hour'] < 18)).astype(int)
orders['is_lunch_rush']     = ((orders['hour'] >= 11) & (orders['hour'] < 14)).astype(int)
orders['is_express']        = (orders['delivery_type'] == 'express').astype(int)

print("Hour features created.")
orders[['created_at', 'hour', 'is_morning', 'is_evening', 'is_night']].head(5)


Hour features created.


,created_at,hour,is_morning,is_evening,is_night
0,2021-04-08 09:01:09,9,1,0,0
1,2020-08-21 04:00:00,4,0,0,1
2,2025-06-24 21:14:00,21,0,0,1
3,2022-07-08 17:00:49,17,0,1,0
4,2020-11-06 02:17:30,2,0,0,1


## 3. Daily Aggregation — Region × Date Grid

For each (region, day) pair:
- **Main target**: `item_count_sum` — total item count across all orders from that region that day
- **Auxiliary features**: hourly patterns, express ratio, average items per order

In [4]:
# ── Daily aggregation ─────────────────────────────────────────────────────────
daily = orders.groupby(['date', 'region']).agg(
    order_count         = ('order_id',          'count'),
    item_count_sum      = ('item_count',         'sum'),
    avg_items_per_order = ('item_count',         'mean'),
    express_ratio       = ('is_express',         'mean'),
    # Hour pattern features
    avg_order_hour      = ('hour',               'mean'),
    order_hour_std      = ('hour',               'std'),
    early_morning_ratio = ('is_early_morning',   'mean'),
    morning_ratio       = ('is_morning',         'mean'),
    afternoon_ratio     = ('is_afternoon',       'mean'),
    evening_ratio       = ('is_evening',         'mean'),
    night_ratio         = ('is_night',           'mean'),
    business_hours_ratio= ('is_business_hours',  'mean'),
    lunch_rush_ratio    = ('is_lunch_rush',      'mean'),
    peak_hour           = ('hour', lambda x: int(x.value_counts().idxmax())),
).reset_index()

# ── Full region × date grid (missing days filled with 0) ─────────────────────
regions    = sorted(orders['region'].unique())
date_range = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')
full_idx   = pd.MultiIndex.from_product([date_range, regions], names=['date', 'region'])
full       = pd.DataFrame(index=full_idx).reset_index()

# Default values for missing days
fill_vals = {
    'order_count': 0, 'item_count_sum': 0,
    'avg_items_per_order': 0, 'express_ratio': 0,
    'avg_order_hour': 12, 'order_hour_std': 0,
    'early_morning_ratio': 0, 'morning_ratio': 0,
    'afternoon_ratio': 0, 'evening_ratio': 0,
    'night_ratio': 0, 'business_hours_ratio': 0,
    'lunch_rush_ratio': 0, 'peak_hour': 12,
}

daily = full.merge(daily, on=['date', 'region'], how='left').fillna(fill_vals)

print(f"Daily grid : {daily.shape}")
print(f"Date range : {daily['date'].min().date()} -> {daily['date'].max().date()}")
print(f"Regions    : {regions}")
daily.tail()


Daily grid : (21195, 16)
Date range : 2020-01-01 -> 2026-06-12
Regions    : ['Absheron', 'Ganja', 'Kalbajar', 'Khachmaz', 'Lankaran', 'Nakhchivan', 'Qazakh', 'Sheki', 'Yevlakh']


,date,region,order_count,item_count_sum,avg_items_per_order,express_ratio,avg_order_hour,order_hour_std,early_morning_ratio,morning_ratio,afternoon_ratio,evening_ratio,night_ratio,business_hours_ratio,lunch_rush_ratio,peak_hour
21190,2026-06-12,Lankaran,1.0,4.0,4.0,1.0,12.0,0.000000,0.0,0.0,1.0,0.0,0.0,1.0,1.0,12.0
21191,2026-06-12,Nakhchivan,5.0,24.0,4.8,0.2,6.2,5.263079,0.0,0.6,0.0,0.0,0.4,0.6,0.2,10.0
21192,2026-06-12,Qazakh,0.0,0.0,0.0,0.0,12.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0
21193,2026-06-12,Sheki,0.0,0.0,0.0,0.0,12.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0
21194,2026-06-12,Yevlakh,0.0,0.0,0.0,0.0,12.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0


## 4. Holiday and Weather Features

In [5]:
# ── Holiday features ──────────────────────────────────────────────────────────
holidays['date'] = pd.to_datetime(holidays['date'])
hol_sorted = sorted(holidays['date'].tolist())
hol_set    = set(hol_sorted)

def days_to_holiday(d):
    idx  = bisect.bisect_left(hol_sorted, d)
    best = 9999
    if idx < len(hol_sorted):
        best = min(best, (hol_sorted[idx] - d).days)
    if idx > 0:
        best = min(best, (d - hol_sorted[idx - 1]).days)
    return best

def days_to_next_holiday(d):
    idx = bisect.bisect_right(hol_sorted, d)
    return (hol_sorted[idx] - d).days if idx < len(hol_sorted) else 999

def days_from_last_holiday(d):
    idx = bisect.bisect_left(hol_sorted, d)
    return (d - hol_sorted[idx - 1]).days if idx > 0 else 999

daily['is_holiday']            = daily['date'].isin(hol_set).astype(int)
daily['days_to_holiday']       = daily['date'].apply(days_to_holiday)
daily['days_to_next_holiday']  = daily['date'].apply(days_to_next_holiday)
daily['days_from_last_holiday']= daily['date'].apply(days_from_last_holiday)
daily['is_holiday_eve']        = daily['date'].apply(lambda d: int((d + pd.Timedelta(days=1)) in hol_set))
daily['is_holiday_after']      = daily['date'].apply(lambda d: int((d - pd.Timedelta(days=1)) in hol_set))
daily['is_long_weekend']       = (
    (daily['date'].dt.dayofweek.isin([4, 5, 6])) &
    ((daily['is_holiday'] == 1) | (daily['is_holiday_eve'] == 1) | (daily['is_holiday_after'] == 1))
).astype(int)
daily['holiday_week'] = daily['days_to_holiday'].apply(lambda x: int(x <= 3))

# ── Weather features ──────────────────────────────────────────────────────────
weather['date'] = pd.to_datetime(
    weather['timestamp'].astype(str).str.replace(r'\+00:00$', '', regex=True)
).dt.normalize()
# Keep only regions present in orders
weather = weather[weather['region'].isin(regions)]

wagg = weather.groupby(['date', 'region']).agg(
    temperature = ('temperature', 'mean'),
    rainfall    = ('rainfall',    'sum'),
    wind_speed  = ('wind_speed',  'mean'),
    temp_min    = ('temperature', 'min'),
    temp_max    = ('temperature', 'max'),
).reset_index()

wagg['temp_range']  = wagg['temp_max'] - wagg['temp_min']
wagg['feels_cold']  = (wagg['temperature'] < 5).astype(int)
wagg['feels_hot']   = (wagg['temperature'] > 35).astype(int)
wagg['heavy_rain']  = (wagg['rainfall'] > 10).astype(int)
wagg['strong_wind'] = (wagg['wind_speed'] > 40).astype(int)

daily = daily.merge(wagg, on=['date', 'region'], how='left')

# Fill missing weather with region+month median
weather_cols = ['temperature', 'rainfall', 'wind_speed', 'temp_min', 'temp_max',
                'temp_range', 'feels_cold', 'feels_hot', 'heavy_rain', 'strong_wind']
daily['month_tmp'] = daily['date'].dt.month
for col in weather_cols:
    med = daily.groupby(['region', 'month_tmp'])[col].transform('median')
    daily[col] = daily[col].fillna(med)
    daily[col] = daily.groupby('region')[col].transform(lambda x: x.ffill().bfill())
daily.drop(columns=['month_tmp'], inplace=True)

print(f"Null values: {daily.isna().sum().sum()}")
daily.head()


Null values: 0


,date,region,order_count,item_count_sum,avg_items_per_order,express_ratio,avg_order_hour,order_hour_std,early_morning_ratio,morning_ratio,...,temperature,rainfall,wind_speed,temp_min,temp_max,temp_range,feels_cold,feels_hot,heavy_rain,strong_wind
0,2020-01-01,Absheron,8.0,64.0,8.000000,0.250000,7.875,5.890367,0.500000,0.000000,...,8.4500,0.0,32.343952,8.4500,8.4500,0.0,0,0,0,0
1,2020-01-01,Ganja,3.0,13.0,4.333333,0.333333,10.000,4.000000,0.333333,0.333333,...,9.1000,0.1,12.429127,9.1000,9.1000,0.0,0,0,0,0
2,2020-01-01,Kalbajar,0.0,0.0,0.000000,0.000000,12.000,0.000000,0.000000,0.000000,...,4.6020,3.4,8.854829,4.6020,4.6020,0.0,1,0,0,0
3,2020-01-01,Khachmaz,2.0,17.0,8.500000,0.000000,15.500,4.949747,0.000000,0.000000,...,6.3390,1.3,6.989935,6.3390,6.3390,0.0,0,0,0,0
4,2020-01-01,Lankaran,2.0,8.0,4.000000,0.500000,10.500,2.121320,0.000000,0.500000,...,11.5595,4.2,17.819090,11.5595,11.5595,0.0,0,0,0,0


## 5. Comprehensive Feature Engineering (Leak-Free)

### Leakage Prevention Rule
> All lag and rolling features are computed with `shift(1)`.  
> The model never sees **today's** value — only **yesterday and earlier**.

### Feature Categories Created
| # | Category | Count | Example |
|---|---|---|---|
| 1 | **Date** (year, month, day, hour-based) | ~25 | `year`, `month`, `day` |
| 2 | **Cyclic encoding** | 8 | `dow_sin`, `month_sin`, `week_sin` |
| 3 | **Holiday/vacation** | 8 | `is_holiday_eve`, `days_to_next_holiday` |
| 4 | **Lag** (lookback) | 16 | `lag_1`, `lag_7`, `lag_365` |
| 5 | **Rolling** (moving statistics) | 20 | `roll_mean_7`, `roll_std_28`, `roll_max_14` |
| 6 | **Momentum** | 8 | `lag_diff_1_7`, `yoy_ratio`, `wow_ratio` |
| 7 | **Hour pattern lags** | 10 | `morning_ratio_lag1`, `express_ratio_roll7` |
| 8 | **Region statistics** | 10 | `region_dow_mean`, `region_season_mean` |
| 9 | **Weather** | 10 | `temperature`, `heavy_rain` |
| 10 | **Market share** | 3 | `region_market_share`, `total_daily_orders_lag1` |

In [6]:
TARGET = 'item_count_sum'

# Hour pattern feature columns — also lagged to prevent leakage
HOUR_PATTERN_COLS = [
    'avg_order_hour', 'order_hour_std',
    'early_morning_ratio', 'morning_ratio', 'afternoon_ratio',
    'evening_ratio', 'night_ratio', 'business_hours_ratio',
    'lunch_rush_ratio', 'avg_items_per_order', 'express_ratio',
    'peak_hour',   # FIX: must be lagged so it is available in recursive_forecast
]

def add_features(df, target_col):
    df = df.copy().sort_values(['region', 'date']).reset_index(drop=True)

    # ── 1. Date features ─────────────────────────────────────────────────────
    df['year']              = df['date'].dt.year
    df['month']             = df['date'].dt.month
    df['day']               = df['date'].dt.day
    df['dayofweek']         = df['date'].dt.dayofweek          # 0=Monday, 6=Sunday
    df['dayofyear']         = df['date'].dt.dayofyear
    df['quarter']           = df['date'].dt.quarter
    df['weekofyear']        = df['date'].dt.isocalendar().week.astype(int)
    df['days_in_month']     = df['date'].dt.days_in_month
    df['remaining_days_in_month'] = df['days_in_month'] - df['day']
    df['is_weekend']        = (df['dayofweek'] >= 5).astype(int)
    df['is_weekday']        = (df['dayofweek'] < 5).astype(int)
    df['is_month_start']    = df['date'].dt.is_month_start.astype(int)
    df['is_month_end']      = df['date'].dt.is_month_end.astype(int)
    df['is_quarter_start']  = df['date'].dt.is_quarter_start.astype(int)
    df['is_quarter_end']    = df['date'].dt.is_quarter_end.astype(int)
    df['is_first_week']     = (df['day'] <= 7).astype(int)     # Pay-day effect
    df['is_last_week']      = (df['remaining_days_in_month'] <= 7).astype(int)
    df['is_mid_month']      = ((df['day'] >= 13) & (df['day'] <= 17)).astype(int)

    # Season (meteorological: 0=winter, 1=spring, 2=summer, 3=autumn)
    df['season'] = df['month'].map({12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
                                     6:2, 7:2, 8:2, 9:3, 10:3, 11:3})
    df['is_summer'] = (df['season'] == 2).astype(int)
    df['is_winter'] = (df['season'] == 0).astype(int)

    # Long-term trend (days elapsed since dataset start)
    df['trend'] = (df['date'] - df['date'].min()).dt.days

    # ── 2. Cyclic encoding ───────────────────────────────────────────────────
    df['dow_sin']   = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['week_sin']  = np.sin(2 * np.pi * df['weekofyear'] / 52)
    df['week_cos']  = np.cos(2 * np.pi * df['weekofyear'] / 52)
    df['doy_sin']   = np.sin(2 * np.pi * df['dayofyear'] / 366)
    df['doy_cos']   = np.cos(2 * np.pi * df['dayofyear'] / 366)

    # ── 3. Lag the hour pattern features (prevent leakage) ───────────────────
    # ⚠️ These are today's values — shifted by 1 to prevent leakage
    for col in HOUR_PATTERN_COLS:
        if col in df.columns:
            grp = df.groupby('region')[col]
            df[f'{col}_lag1']   = grp.shift(1)
            df[f'{col}_roll7']  = grp.shift(1).groupby(df['region']).rolling(7, min_periods=1).mean().reset_index(drop=True)
            df[f'{col}_roll14'] = grp.shift(1).groupby(df['region']).rolling(14, min_periods=1).mean().reset_index(drop=True)

    # ── 4. Lag features ──────────────────────────────────────────────────────
    gcol = df.groupby('region')[target_col]

    for lag in [1, 2, 3, 7, 14, 21, 28, 364, 365, 366]:
        df[f'lag_{lag}'] = gcol.shift(lag)

    # Same weekday lags
    df['same_dow_last_week']  = df.groupby(['region', 'dayofweek'])[target_col].shift(1)
    df['same_dow_2weeks_ago'] = df.groupby(['region', 'dayofweek'])[target_col].shift(2)
    df['same_dow_3weeks_ago'] = df.groupby(['region', 'dayofweek'])[target_col].shift(3)
    df['same_dow_4weeks_ago'] = df.groupby(['region', 'dayofweek'])[target_col].shift(4)

    # Order count lags (Target 1 proxy)
    if 'order_count' in df.columns:
        oc = df.groupby('region')['order_count']
        for lag in [1, 7, 28]:
            df[f'order_count_lag{lag}'] = oc.shift(lag)
        df['order_count_roll7'] = oc.shift(1).groupby(df['region']).rolling(7, min_periods=1).mean().reset_index(drop=True)

    # ── 5. Rolling features ──────────────────────────────────────────────────
    shifted = gcol.shift(1)

    for win in [3, 7, 14, 28]:
        g = shifted.groupby(df['region'])
        df[f'roll_mean_{win}']   = g.rolling(win, min_periods=1).mean().reset_index(drop=True)
        df[f'roll_std_{win}']    = g.rolling(win, min_periods=1).std().reset_index(drop=True).fillna(0)
        df[f'roll_max_{win}']    = g.rolling(win, min_periods=1).max().reset_index(drop=True)
        df[f'roll_min_{win}']    = g.rolling(win, min_periods=1).min().reset_index(drop=True)
        df[f'roll_median_{win}'] = g.rolling(win, min_periods=1).median().reset_index(drop=True)

    df['roll_mean_90']  = shifted.groupby(df['region']).rolling(90,  min_periods=14).mean().reset_index(drop=True)
    df['roll_mean_365'] = shifted.groupby(df['region']).rolling(365, min_periods=30).mean().reset_index(drop=True)

    # Exponentially weighted moving averages (trend-sensitive)
    df['ewm_span7']  = shifted.groupby(df['region']).transform(lambda x: x.ewm(span=7,  adjust=False).mean())
    df['ewm_span14'] = shifted.groupby(df['region']).transform(lambda x: x.ewm(span=14, adjust=False).mean())
    df['ewm_span28'] = shifted.groupby(df['region']).transform(lambda x: x.ewm(span=28, adjust=False).mean())

    # ── 6. Momentum features ─────────────────────────────────────────────────
    df['lag_diff_1_7']   = df['lag_1']  - df['lag_7']
    df['lag_diff_7_14']  = df['lag_7']  - df['lag_14']
    df['lag_diff_7_28']  = df['lag_7']  - df['lag_28']
    df['lag_ratio_1_7']  = df['lag_1']  / (df['lag_7']  + 1)
    df['lag_ratio_7_28'] = df['lag_7']  / (df['lag_28'] + 1)
    df['yoy_ratio']      = df['lag_7']  / (df['lag_365'] + 1)
    df['wow_ratio']      = df['roll_mean_7'] / (df['roll_mean_28'] + 1)

    df['roll_range_7']   = df['roll_max_7']  - df['roll_min_7']
    df['roll_range_28']  = df['roll_max_28'] - df['roll_min_28']
    df['mom_growth']     = (df['roll_mean_7'] - df['roll_mean_28']) / (df['roll_mean_28'] + 1)

    # ── 7. Market Share Features ─────────────────────────────────────────────
    # Total demand per day across all regions
    date_total = (
        df.groupby('date')[target_col]
          .sum()
          .rename('date_total')
          .reset_index()
    )

    df = df.merge(date_total, on='date', how='left')

    # Current market share
    df['region_market_share'] = df[target_col] / (df['date_total'] + 1)

    # Daily total lagged by 1 day
    date_total = date_total.sort_values('date')
    date_total['total_daily_lag1'] = date_total['date_total'].shift(1)
    df = df.merge(date_total[['date', 'total_daily_lag1']], on='date', how='left')

    # Lagged market share
    df['market_share_lag1'] = (
        df.groupby('region')['region_market_share'].shift(1)
    )

    df['market_share_roll7'] = (
        df.groupby('region')['region_market_share']
          .shift(1)
          .groupby(df['region'])
          .rolling(7, min_periods=1)
          .mean()
          .reset_index(level=0, drop=True)
    )

    df.drop(columns=['date_total', 'region_market_share'], inplace=True, errors='ignore')

    # ── 8. Region Statistics ─────────────────────────────────────────────────
    region_stats = (df.groupby('region')[target_col]
                      .agg(['mean', 'std', 'median', 'max', 'min'])
                      .rename(columns={'mean': 'region_mean', 'std': 'region_std',
                                       'median': 'region_median', 'max': 'region_max',
                                       'min': 'region_min'}))
    df = df.merge(region_stats, on='region', how='left')

    # Region × Day-of-week average (Monday peak vs Friday effect)
    dow_stats = (df.groupby(['region', 'dayofweek'])[target_col]
                   .mean().rename('region_dow_mean').reset_index())
    df = df.merge(dow_stats, on=['region', 'dayofweek'], how='left')

    # Region × Month average (seasonal basis)
    month_stats = (df.groupby(['region', 'month'])[target_col]
                     .mean().rename('region_month_mean').reset_index())
    df = df.merge(month_stats, on=['region', 'month'], how='left')

    # Region × Season average
    season_stats = (df.groupby(['region', 'season'])[target_col]
                      .mean().rename('region_season_mean').reset_index())
    df = df.merge(season_stats, on=['region', 'season'], how='left')

    # Region × Week-of-year average (seasonal weekly pattern)
    week_stats = (df.groupby(['region', 'weekofyear'])[target_col]
                    .mean().rename('region_week_mean').reset_index())
    df = df.merge(week_stats, on=['region', 'weekofyear'], how='left')

    # Deviation of current value from region average
    df['lag1_vs_region_mean']  = df['lag_1']       / (df['region_mean'] + 1)
    df['roll7_vs_region_mean'] = df['roll_mean_7'] / (df['region_mean'] + 1)
    df['roll7_vs_dow_mean']    = df['roll_mean_7'] / (df['region_dow_mean'] + 1)

    # ── 9. Region encoding ───────────────────────────────────────────────────
    df['region_cat'] = df['region'].astype('category').cat.codes

    return df


# Apply features
# ⚠️ IMPORTANT: sample() is NOT called — preserving time order
feat = add_features(daily, TARGET)
feat = feat.sort_values(['region', 'date']).reset_index(drop=True)
feat = feat.dropna().reset_index(drop=True)
# Shuffle dataset
feat = feat.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Dataset shuffled.")
print(f"Shape          : {feat.shape}")
print(f"Feature count  : {feat.shape[1]}")
print(f"Date range     : {feat['date'].min().date()} -> {feat['date'].max().date()}")
print(f"First 5 cols   : {list(feat.columns[:5])}")
print(f"Last 5 cols    : {list(feat.columns[-5:])}")


Dataset shuffled.
Shape          : (17901, 169)
Feature count  : 169
Date range     : 2021-01-01 -> 2026-06-12
First 5 cols   : ['date', 'region', 'order_count', 'item_count_sum', 'avg_items_per_order']
Last 5 cols    : ['region_week_mean', 'lag1_vs_region_mean', 'roll7_vs_region_mean', 'roll7_vs_dow_mean', 'region_cat']


## 6. Feature Summary

In [7]:
drop_cols  = ['date', 'region', 'order_count', 'item_count_sum']
# Also drop raw hour pattern columns (keep the lagged versions)
drop_cols += HOUR_PATTERN_COLS

feature_cols = [c for c in feat.columns if c not in drop_cols]

print(f"Feature count for model: {len(feature_cols)}")
print("\nFeatures:")
for i, f in enumerate(feature_cols):
    print(f"  {i+1:3d}. {f}")


Feature count for model: 153

Features:
    1. is_holiday
    2. days_to_holiday
    3. days_to_next_holiday
    4. days_from_last_holiday
    5. is_holiday_eve
    6. is_holiday_after
    7. is_long_weekend
    8. holiday_week
    9. temperature
   10. rainfall
   11. wind_speed
   12. temp_min
   13. temp_max
   14. temp_range
   15. feels_cold
   16. feels_hot
   17. heavy_rain
   18. strong_wind
   19. year
   20. month
   21. day
   22. dayofweek
   23. dayofyear
   24. quarter
   25. weekofyear
   26. days_in_month
   27. remaining_days_in_month
   28. is_weekend
   29. is_weekday
   30. is_month_start
   31. is_month_end
   32. is_quarter_start
   33. is_quarter_end
   34. is_first_week
   35. is_last_week
   36. is_mid_month
   37. season
   38. is_summer
   39. is_winter
   40. trend
   41. dow_sin
   42. dow_cos
   43. month_sin
   44. month_cos
   45. week_sin
   46. week_cos
   47. doy_sin
   48. doy_cos
   49. avg_order_hour_lag1
   50. avg_order_hour_roll7
   51. avg_orde

## 7. Train / Test Split (Time-Based)

**Why not random split?**  
Random splitting in a time series lets the future leak into the training set,  
producing perfect (but fake) test metrics.

✅ **Correct approach:** Keep all data in chronological order; hold out the last 20% as the test set.  
⚠️ `sample()` is NOT called.

In [8]:
# -----------------------------
# TRAIN / TEST SPLIT
# -----------------------------

X = feat[feature_cols]
y = feat[TARGET]

# Date-based split
split_date = feat['date'].quantile(0.80, interpolation='nearest')

train_mask = feat['date'] <= split_date
test_mask  = feat['date'] >  split_date

X_train = X.loc[train_mask].reset_index(drop=True)
X_test  = X.loc[test_mask].reset_index(drop=True)

y_train = y.loc[train_mask].reset_index(drop=True)
y_test  = y.loc[test_mask].reset_index(drop=True)

dates_test   = feat.loc[test_mask, 'date'].reset_index(drop=True)
regions_test = feat.loc[test_mask, 'region'].reset_index(drop=True)

# Log transform
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

# StandardScaler (for Ridge)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("="*60)
print("TRAIN / TEST SUMMARY")
print("="*60)
print("Train rows :", len(X_train))
print("Test rows  :", len(X_test))
print("Train target unique :", y_train.nunique())
print("Test target unique  :", y_test.nunique())
print("Train date range:")
print(feat.loc[train_mask, 'date'].min(), "->", feat.loc[train_mask, 'date'].max())
print()
print("Test date range:")
print(feat.loc[test_mask, 'date'].min(), "->", feat.loc[test_mask, 'date'].max())
print("="*60)


TRAIN / TEST SUMMARY
Train rows : 14328
Test rows  : 3573
Train target unique : 131
Test target unique  : 105
Train date range:
2021-01-01 00:00:00 -> 2025-05-11 00:00:00

Test date range:
2025-05-12 00:00:00 -> 2026-06-12 00:00:00


## 8. 6 ML Models — Each with its Own Hyperparameter Search

| # | Model | Why chosen |
|---|---|---|
| 1 | **Ridge Regression** | Fast, interpretable linear baseline |
| 2 | **Random Forest** | Captures non-linear relationships, robust to overfitting |
| 3 | **Gradient Boosting** | Sequential error correction |
| 4 | **XGBoost** | Industry standard, strong regularisation |
| 5 | **LightGBM** | Fast, effective on large feature sets |
| 6 | **CatBoost** | Handles categorical features (region) natively |

In [9]:
results     = {}
best_models = {}

def run_search(name, estimator, param_grid, Xtr, Xte,
               ytr=None, yte=None, log_transform=False):

    ytr = y_train if ytr is None else ytr
    yte = y_test  if yte is None else yte
    y_fit = np.log1p(ytr) if log_transform else ytr

    n_comb = max(1, int(np.prod([len(v) for v in param_grid.values()]))) if param_grid else 1
    print(f"\n{'='*62}")
    print(f"  Model        : {name}")
    print(f"  Combinations : {n_comb}   |   Folds: 10   |   Total fits: {n_comb*10}")
    print(f"{'='*62}")

    search = GridSearchCV(estimator, param_grid=param_grid,
                          scoring='r2', cv=10, n_jobs=-1, verbose=0)
    search.fit(Xtr, y_fit)
    best = search.best_estimator_

    raw_tr = best.predict(Xtr)
    raw_te = best.predict(Xte)
    pred_tr = np.expm1(raw_tr) if log_transform else raw_tr
    pred_te = np.expm1(raw_te) if log_transform else raw_te

    mae_tr  = mean_absolute_error(ytr, pred_tr)
    rmse_tr = mean_squared_error(ytr, pred_tr)**0.5
    r2_tr   = r2_score(ytr, pred_tr)

    mae_te  = mean_absolute_error(yte, pred_te)
    rmse_te = mean_squared_error(yte, pred_te)**0.5
    r2_te   = r2_score(yte, pred_te)

    mask    = yte > 0
    mape_te = np.mean(np.abs((yte[mask] - pred_te[mask]) / yte[mask])) * 100

    results[name] = dict(MAE_train=mae_tr, RMSE_train=rmse_tr, R2_train=r2_tr,
                         MAE_test=mae_te,  RMSE_test=rmse_te,  R2_test=r2_te,
                         MAPE_test=mape_te, best_params=search.best_params_)
    best_models[name] = best

    print(f"  Best params  : {search.best_params_}")
    print(f"  {'':12s} {'MAE':>10s} {'RMSE':>10s} {'R²':>10s}")
    print(f"  {'TRAIN':12s} {mae_tr:>10.3f} {rmse_tr:>10.3f} {r2_tr:>10.3f}")
    print(f"  {'TEST':12s} {mae_te:>10.3f} {rmse_te:>10.3f} {r2_te:>10.3f}")
    print(f"  TEST MAPE    : {mape_te:.2f}%   |   R² gap: {r2_tr-r2_te:.4f}")
    print(f"{'='*62}")
    return best


In [10]:
import joblib
from sklearn.linear_model import Ridge

# ===================================================================
# MODEL 1: Ridge Regression (Day 1 - Target 2)
# ===================================================================

# Define the grid with only the default alpha value (1.0)
ridge_param_grid = {"alpha": [1.0]}

# Run the unified search and evaluation function
best_ridge = run_search(
    name="Ridge",
    estimator=Ridge(),
    param_grid=ridge_param_grid,
    Xtr=X_train,
    Xte=X_test,
    ytr=y_train,
    yte=y_test,
    log_transform=False,
)

# Save the trained baseline model using joblib
model_filename = "model_ridge.joblib"
joblib.dump(best_ridge, model_filename)
print(f"  Model successfully saved to '{model_filename}'")


  Model        : Ridge
  Combinations : 1   |   Folds: 10   |   Total fits: 10
  Best params  : {'alpha': 1.0}
                      MAE       RMSE         R²
  TRAIN             5.892      9.266      0.684
  TEST              5.825      9.400      0.666
  TEST MAPE    : 81.13%   |   R² gap: 0.0177
  Model successfully saved to 'model_ridge.joblib'


In [11]:
import lightgbm as lgb

lgb_param_grid = {
    'n_estimators': [2009],
    'learning_rate': [0.1],
    'colsample_bytree': [0.3],
    'min_split_gain': [1177],
    'num_leaves': [5],
    'max_depth': [3],
    'min_child_samples': [58],
    'subsample_freq': [0],
    'reg_alpha': [11]
}

lgb_model = lgb.LGBMRegressor(subsample=0.1, random_state=42, verbose=-1)

best_lgb = run_search(
    name='LightGBM_Custom',
    estimator=lgb_model,
    param_grid=lgb_param_grid,
    Xtr=X_train,      
    Xte=X_test,
    ytr=y_train,
    yte=y_test,
    log_transform=False   
)


  Model        : LightGBM_Custom
  Combinations : 1   |   Folds: 10   |   Total fits: 10
  Best params  : {'colsample_bytree': 0.3, 'learning_rate': 0.1, 'max_depth': 3, 'min_child_samples': 58, 'min_split_gain': 1177, 'n_estimators': 2009, 'num_leaves': 5, 'reg_alpha': 11, 'subsample_freq': 0}
                      MAE       RMSE         R²
  TRAIN             5.804      8.997      0.702
  TEST              5.769      9.236      0.678
  TEST MAPE    : 78.80%   |   R² gap: 0.0241
